# 6. `tfx_g_soft` — tfx_g + 노브 loss 개선 (주변 칸에도 점수: 가우시안 soft label σ=2칸 + EMD 거리 벌점)

- 설정: `testFx/configs/tfx_g_soft.yaml` · 스크립트: `testFx/train_tfx.py`
- tfx_g와 loss만 다름 -> tfx_g와 비교. 로그의 `ce_hard`가 다른 실험의 loss와 같은 기준.
- 결과 위치 (Drive `26_Research/myFX/`): `ckpt/tfx_g_soft/`, `logs/tfx_g_soft/`, `eval/tfx_g_soft/`
- 런타임 → 런타임 유형 변경 → **GPU (L4 권장)**. 끊기면 0)과 2) 셀만 다시 실행하면 `latest_checkpoint.pt`에서 이어서 학습.
- 다른 실험 노트북과 **동시에** 돌려도 됨 (Drive 폴더가 서로 다름). 대신 컴퓨팅 단위가 그만큼 빨리 줄어듦.
- 모든 실험이 끝나면 `5_compare.ipynb`에서 같이 비교.

In [ ]:
NAME = "tfx_g_soft"

### 0) 준비 (런타임을 새로 연결할 때마다 위에서부터 실행)

In [ ]:
# ---- 경로 (모든 노트북이 같은 값을 써야 결과가 한 곳에 모임) ----------------
REPO_URL    = "https://github.com/kuhberaBubo/myFX.git"
STEMFX_URL  = "https://github.com/barry-mir/stemfx.git"
RESEARCH    = "/content/drive/MyDrive/26_Research"
DATASET_ZIP = f"{RESEARCH}/dataset.zip"
DRIVE_ROOT  = f"{RESEARCH}/myFX"                       # 결과 전부 여기 (ckpt/ logs/ eval/)
# ft_v1 (지난번 결과)이 DRIVE_ROOT/ckpt/ft_v1에 없으면 예전 위치에서 찾음
OLD_CKPT = {"ft_v1": "/content/drive/MyDrive/myFX/ckpt/ft_v1"}
# -------------------------------------------------------------------------
import os
CODE_DIR  = "/content/myFX"
EXPERIMENTS = {
    "ft_frozen": ("ft_stemfx/train.py", "ft_stemfx/configs/ft_frozen.yaml"),
    "ft_full":   ("ft_stemfx/train.py", "ft_stemfx/configs/ft_full.yaml"),
    "tfx_g":     ("testFx/train_tfx.py", "testFx/configs/tfx_g.yaml"),
    "tfx_mert":  ("testFx/train_tfx.py", "testFx/configs/tfx_mert.yaml"),
    "tfx_g_soft": ("testFx/train_tfx.py", "testFx/configs/tfx_g_soft.yaml"),
}
def ckpt_dir(name):
    new, old = f"{DRIVE_ROOT}/ckpt/{name}", OLD_CKPT.get(name)
    return old if old and not os.path.isdir(new) and os.path.isdir(old) else new
log_dir  = lambda name: f"{DRIVE_ROOT}/logs/{name}"
eval_dir = lambda name: f"{DRIVE_ROOT}/eval/{name}"
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "no GPU"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, shutil, subprocess
try:
    from google.colab import userdata
    tok = userdata.get("GH_TOKEN")
except Exception:
    tok = None
url = REPO_URL.replace("https://", f"https://{tok}@") if tok else REPO_URL
if os.path.isdir(CODE_DIR) and not os.path.isdir(f"{CODE_DIR}/.git"):
    shutil.rmtree(CODE_DIR)
if os.path.isdir(f"{CODE_DIR}/.git"):
    !git -C {CODE_DIR} pull --ff-only
else:
    r = subprocess.run(["git", "clone", "-q", url, CODE_DIR], env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
    if r.returncode != 0:
        raise SystemExit("git clone failed (private repo -> Colab secret GH_TOKEN)")
if not os.path.isfile(f"{CODE_DIR}/stemfx/stemfx/model.py"):
    !git clone -q {STEMFX_URL} {CODE_DIR}/stemfx
!git -C {CODE_DIR} log -1 --oneline

In [ ]:
# stemfx pins numpy<2 -> no deps. transformers + nnAudio: MERT
!pip install -q -e {CODE_DIR}/stemfx --no-deps
!pip install -q multiafx pyloudnorm huggingface_hub soundfile transformers nnAudio
import numpy, torch, transformers
print("numpy", numpy.__version__, "| torch", torch.__version__, "| transformers", transformers.__version__,
      "| bf16", torch.cuda.is_available() and torch.cuda.is_bf16_supported())

In [ ]:
# dataset zip -> local disk (Drive에서 직접 읽으면 느림)
import glob
def find_root():
    hits = sorted(glob.glob("/content/run_20k/train.jsonl") + glob.glob("/content/*/run_20k/train.jsonl"))
    return os.path.dirname(hits[0]) if hits else None
if find_root() is None:
    !cp "{DATASET_ZIP}" /content/dataset.zip
    !unzip -q -o /content/dataset.zip -d /content && rm /content/dataset.zip
DATA_ROOT = find_root()
assert DATA_ROOT, "train.jsonl not found after unzip"
print("DATA_ROOT =", DATA_ROOT)

In [ ]:
# ---- helpers --------------------------------------------------------------
import json
def run_exp(name, quick=False, extra=""):
    script, cfg = EXPERIMENTS[name]
    if quick:   # 64/32 samples, 1 epoch, throwaway folder
        dirs = (f"train.checkpoint_dir=/content/tmp/{name}/ckpt train.log_dir=/content/tmp/{name}/logs "
                "data.max_train_samples=64 data.max_val_samples=32 train.num_epochs=1 train.resume=null")
        shutil.rmtree(f"/content/tmp/{name}", ignore_errors=True)
    else:
        dirs = f"train.checkpoint_dir={ckpt_dir(name)} train.log_dir={log_dir(name)}"
    cmd = f"python {CODE_DIR}/{script} --config {CODE_DIR}/{cfg} data.root={DATA_ROOT} {dirs} {extra}"
    print("$", cmd)
    get_ipython().system(cmd)

def status(names=None):
    names = names or ["ft_v1"] + list(EXPERIMENTS)
    print(f"{'exp':10s} {'epoch':>9s} {'best':>8s} {'val loss':>9s} {'val mae':>8s} {'base':>6s} {'min/ep':>7s}  state")
    for name in names:
        p = f"{log_dir(name)}/status.json"
        if not os.path.isfile(p):
            ck = os.path.isfile(f"{ckpt_dir(name)}/best_model.pt")
            print(f"{name:10s} {'-':>9s}   {'(best_model.pt 있음, status 없음)' if ck else ''}"); continue
        s = json.load(open(p))
        state = ("early-stopped" if s["early_stopped"] else
                 "done" if s["finished"] else f"{s['bad_epochs']} w/o improvement")
        print(f"{name:10s} {s['epoch']:>4d}/{s['num_epochs']:<4d} {s['best_val']:8.4f} {s['val']['loss']:9.4f} "
              f"{s['val']['mae_val']:8.4f} {s['baseline_mae']:6.3f} {s['epoch_sec']/60:7.1f}  {state}")

N_EVAL, EVAL_SEED = 100, 0      # 모든 모델이 같은 val 100개를 예측 (바꾸지 말 것)
def predict(name, device="cuda", value_decode="median"):
    """median (기본): 노브 = 예측 분포의 중앙값 -> eval/<name>/
       argmax: 가장 확률 높은 칸 (StemFX 원래 방식) -> eval/<name>_argmax/"""
    ck = f"{ckpt_dir(name)}/best_model.pt"
    if not os.path.isfile(ck):
        print("skip (no best_model.pt):", name); return False
    out = eval_dir(name) if value_decode == "median" else eval_dir(f"{name}_argmax")
    # 예전(argmax) 예측이 eval/<name>/에 있으면 eval/<name>_argmax/로 옮김
    legacy = eval_dir(name)
    if (os.path.isfile(f"{legacy}/preds.jsonl") and not os.path.isfile(f"{legacy}/predict_meta.json")
            and not os.path.exists(eval_dir(f"{name}_argmax"))):
        shutil.move(legacy, eval_dir(f"{name}_argmax"))
        print(f"moved old argmax predictions -> {eval_dir(name + '_argmax')}")
    get_ipython().system(f"python {CODE_DIR}/ft_stemfx/render_eval.py predict --ckpt {ck} --root {DATA_ROOT} "
                         f"--out {out} --n {N_EVAL} --seed {EVAL_SEED} --device {device} --batch-size 16 "
                         f"--value-decode {value_decode}")
    return True

### 1) 빠른 테스트 (1~3분)
에러가 없는지, 진행바의 `s/it`(한 step 시간)를 확인. 전체 1 epoch = 681 step (batch 16).

In [ ]:
run_exp(NAME, quick=True)

### 2) 학습
early stopping으로 끝나면 자동 종료. 브라우저 탭은 열어둘 것. 진행 상황은 아래 `status` 셀이나 TensorBoard.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {log_dir(NAME)}

In [ ]:
run_exp(NAME)
status([NAME])

In [ ]:
status([NAME])   # 언제든 실행 가능

### 3) 평가용 예측 (GPU, 몇 분)
`best_model.pt`로 공통 val 100개(seed 0)의 체인을 예측 → Drive `eval/tfx_g_soft/preds.jsonl`.
노브 값은 예측 분포의 **중앙값**(기본, `eval/tfx_g_soft/`)과 가장 확률 높은 칸(`argmax`, StemFX 원래 방식,
`eval/tfx_g_soft_argmax/`) 두 가지로 예측해서 비교할 수 있게 함. 이미 한 샘플은 건너뜀.
렌더 없이 체인·노브 지표만 먼저 보려면 다음 셀 `score`. 소리 비교는 데스크톱에서 렌더한 뒤.

In [ ]:
predict(NAME)                          # 노브 = 분포의 중앙값 -> eval/NAME/
predict(NAME, value_decode="argmax")   # 노브 = 가장 확률 높은 칸 -> eval/NAME_argmax/

In [ ]:
get_ipython().system(f"python {CODE_DIR}/ft_stemfx/render_eval.py score --root {DATA_ROOT} --out {eval_dir(NAME)} | tail -n 3")
s = json.load(open(f"{eval_dir(NAME)}/summary.json"))
print(json.dumps(s["chain"], indent=1))
print(f"{'effect':18s} {'F1':>6s} {'knob':>7s} {'base':>7s}")
for e, v in s["per_effect"].items():
    print(f"{e:18s} {v['f1']:6.3f} {v['knob_mae'] or float('nan'):7.4f} {v['knob_mae_baseline'] or float('nan'):7.4f}")